# 유물 파라미터 탐색 v3 — CEM 변분추론 + GPU 병렬화

## v3의 핵심 개선

| 항목 | v1/v2 (랜덤/완전 탐색) | v3 (CEM 변분추론) |
|------|----------------------|------------------|
| 탐색 방식 | 무작위 샘플링 or 완전 탐색 | **분포를 학습하며 수렴** |
| GPU 활용 | 제한적 | **배치 B=100,000 병렬 계산** |
| 수렴 속도 | 느림 | **수십 배 빠름** |
| 3성 보장 | 우연에 의존 | **n_3_100 ≥ 1 하드 제약** |

## 변분추론(CEM) 원리

```
1. 각 파라미터에 균등 확률 분포 초기화
2. 배치 B개 조합 샘플링
3. 전체 B개를 GPU에서 마르코프 체인 병렬 계산
4. 상위 10% (엘리트) 선택
5. 엘리트 분포로 파라미터 분포 업데이트
6. 2~5 반복 → 분포가 최적값으로 수렴
```

## GPU 설정

```
런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU → 저장
```

## 사용 순서

1. GPU 런타임 설정 후 **전체 실행** (런타임 → 모두 실행)
2. Cell 2에서 랭킹 TXT 파일 업로드
3. 결과 CSV + 차트 자동 다운로드

In [ ]:
# ============================================================
# 0. 환경 확인
# ============================================================
import sys, os, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# GPU 확인
try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        get_ipython().system('nvidia-smi')
except Exception as e:
    print('torch 없음:', e)

# CuPy 로드
USE_GPU = False
try:
    import cupy as cp
    _ = cp.arange(3).sum()
    USE_GPU = True
    xp = cp
    print('\nCuPy OK → GPU 배치 연산 활성화')
except Exception as e:
    xp = np
    print('\nCuPy 없음 → NumPy CPU fallback')
    print('  사유:', repr(e))

print('준비 완료 | USE_GPU =', USE_GPU)

In [ ]:
# ============================================================
# 1. 랭킹 TXT 업로드 및 파싱
# ============================================================
# 업로드 파일: 랭킹_및_파라미터.txt  (Replit에서 다운로드)

def parse_ranking_txt(path: str) -> np.ndarray:
    """TXT에서 점수 배열 파싱 (100P 이상)"""
    scores = []
    in_data = False
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            if '순위' in line and '닉네임' in line and '점수' in line:
                in_data = True
                continue
            if not in_data:
                continue
            if not line.strip() or set(line.strip()) <= set('─ ='):
                continue
            if 'END' in line or line.strip().startswith('='):
                break
            parts = line.split()
            if not parts:
                continue
            try:
                int(parts[0])  # 첫 토큰이 순위 숫자인지 확인
                for tok in parts[2:]:
                    clean = tok.replace(',', '').replace('P', '')
                    if clean.isdigit():
                        sc = int(clean)
                        if sc >= 100:
                            scores.append(sc)
                        break
            except (ValueError, IndexError):
                continue
    arr = np.array(sorted(scores, reverse=True), dtype=np.float64)
    return arr

SCORES = None

try:
    from google.colab import files
    print('TXT 파일을 업로드하세요.')
    uploaded = files.upload()
    if uploaded:
        txt_path = list(uploaded.keys())[0]
        SCORES = parse_ranking_txt(txt_path)
        print(f'파싱 완료: {len(SCORES)}명')
        print(f'  최고 {SCORES[0]:,.0f}P | 중앙값 {np.median(SCORES):,.0f}P | 최저 {SCORES[-1]:,.0f}P')
except Exception as e:
    print('업로드 건너뜀:', e)

# 파일이 없으면 하드코딩 데이터 사용
if SCORES is None or len(SCORES) == 0:
    print('\n↓ 하드코딩 데이터 사용 (실제 Production 상위 데이터)')
    SCORES = np.array([
        160060,60395,48891,35768,26313,25862,24263,21607,16499,16399,
        15600,15127,14974,14904,13638,10832,10500,10092,9480,9441,
        9059,9039,8643,8347,7516,7475,7427,7005,6657,6641,
        6610,6593,6316,6296,6288,6281,6276,6276,6263,6190,
        5919,5887,5879,5846,5318,5236,5233,5138,5084,5074,
        5073,5073,5072,5001,4924,4896,4856,4853,4806,4758,
        4714,4658,4639,4610,4583,4476,
        3500,3200,2900,2600,2300,2000,1800,1600,1400,1200,
        1000,800,600,400,200,100
    ], dtype=np.float64)
    print(f'하드코딩: {len(SCORES)}명')

N_USERS = len(SCORES)
print(f'\n총 {N_USERS}명 데이터 준비 완료')

In [ ]:
# ============================================================
# 2. 파라미터 공간 & 사전 계산 테이블
# ============================================================

# ── 파라미터 옵션 (인덱스로 관리) ────────────────────────────────────
#  idx: [c1, c2, c3, pc12, pr12, pc23, pr23, p1, p2, p3, bp]
COST1_OPTS  = np.array([0.5, 0.65, 0.8, 1.0, 1.2, 1.4])
COST2_OPTS  = np.array([0.7, 0.85, 1.0, 1.2, 1.4, 1.6])
COST3_OPTS  = np.array([0.5, 0.7, 0.9, 1.1, 1.4, 1.7, 2.0])
PROMO12C    = np.array([150, 250, 400, 500, 700, 1000], dtype=np.float64)
PROMO12R    = np.array([0.45, 0.55, 0.65, 0.75])
PROMO23C    = np.array([300, 500, 800, 1200, 1800, 2500, 3000], dtype=np.float64)
PROMO23R    = np.array([0.35, 0.45, 0.55, 0.65, 0.75])
PRESET_NAMES = ['easy', 'current', 'mild_hard', 'hard']
BONUS_NAMES  = ['low', 'mid', 'high']

SIZES   = [6, 6, 7, 6, 4, 7, 5, 4, 4, 4, 3]   # 각 파라미터 옵션 수
N_PARAMS = len(SIZES)  # 11
TOTAL_COMBOS = int(np.prod(SIZES))

print(f'파라미터: {N_PARAMS}개, 총 조합: {TOTAL_COMBOS:,}가지')

# ── 확률 테이블 빌드 ─────────────────────────────────────────────────
PROB_DEFS = {
    1: {
        'easy':      [(0,5,92,7,1),(6,15,87,11,2),(16,25,81,14,5),(26,35,75,18,7),(36,45,70,21,9),(46,50,65,23,12)],
        'current':   [(0,5,87,11,2),(6,15,82,13,5),(16,25,76,16,8),(26,35,70,19,11),(36,45,65,21,14),(46,50,60,23,17)],
        'mild_hard': [(0,5,83,13,4),(6,15,78,15,7),(16,25,72,18,10),(26,35,66,21,13),(36,45,61,23,16),(46,50,56,25,19)],
        'hard':      [(0,5,78,15,7),(6,15,73,17,10),(16,25,67,20,13),(26,35,61,23,16),(36,45,56,25,19),(46,50,51,27,22)],
    },
    2: {
        'easy':      [(0,20,88,10,2),(21,40,81,13,6),(41,60,74,18,8),(61,70,67,22,11),(71,80,61,25,14)],
        'current':   [(0,20,84,13,3),(21,40,77,16,7),(41,60,70,20,10),(61,70,63,24,13),(71,80,57,27,16)],
        'mild_hard': [(0,20,80,15,5),(21,40,73,18,9),(41,60,66,22,12),(61,70,59,26,15),(71,80,53,29,18)],
        'hard':      [(0,20,75,18,7),(21,40,68,21,11),(41,60,61,25,14),(61,70,54,28,18),(71,80,48,31,21)],
    },
    3: {
        'easy':      [(0,20,86,11,3),(21,40,79,14,7),(41,60,72,17,11),(61,80,66,21,13),(81,90,61,26,13),(91,100,57,30,13)],
        'current':   [(0,20,82,13,5),(21,40,75,16,9),(41,60,68,19,13),(61,80,62,23,15),(81,90,57,28,15),(91,100,53,32,15)],
        'mild_hard': [(0,20,78,15,7),(21,40,71,18,11),(41,60,64,21,15),(61,80,58,25,17),(81,90,53,30,17),(91,100,49,34,17)],
        'hard':      [(0,20,73,18,9),(21,40,66,21,13),(41,60,59,24,17),(61,80,53,28,19),(81,90,48,33,19),(91,100,44,37,19)],
    }
}

def _build_prob_table(star, max_lv):
    succ = np.zeros((4, max_lv), dtype=np.float64)
    fail = np.zeros((4, max_lv), dtype=np.float64)
    for pi, pname in enumerate(PRESET_NAMES):
        for (lo, hi, s, k, f) in PROB_DEFS[star][pname]:
            tot = float(s + k + f)
            succ[pi, lo:hi+1] = s / tot
            fail[pi, lo:hi+1] = f / tot
    return succ, fail

def _build_cost_base(star, max_lv):
    base = np.zeros(max_lv, dtype=np.float64)
    for n in range(max_lv):
        if star == 1:
            if   n <= 10: b = 2*n
            elif n <= 20: b = 20 + 2*(n-10)
            elif n <= 30: b = 40 + 3*(n-20)
            elif n <= 40: b = 70 + 5*(n-30)
            else:         b = 120 + 7*(n-40)
        elif star == 2:
            if   n <= 40: b = 15 + 3*n
            elif n <= 60: b = 135 + 6*(n-40)
            elif n <= 70: b = 255 + 10*(n-60)
            else:         b = 355 + 15*(n-70)
        else:
            if   n <= 20: b = 40 + 3*n
            elif n <= 40: b = 100 + 5*(n-20)
            elif n <= 60: b = 200 + 7*(n-40)
            elif n <= 80: b = 340 + 11*(n-60)
            elif n <= 90: b = 560 + 16*(n-80)
            else:         b = 720 + 22*(n-90)
        base[n] = b
    return base

# 테이블 빌드
SUCC1, FAIL1 = _build_prob_table(1, 50)
SUCC2, FAIL2 = _build_prob_table(2, 80)
SUCC3, FAIL3 = _build_prob_table(3, 100)
BASE1 = _build_cost_base(1, 50)
BASE2 = _build_cost_base(2, 80)
BASE3 = _build_cost_base(3, 100)

# 성급강화 실패 복귀 가중치
FAIL_W  = np.array([18,16,14,12,10,9,7,6,5,3], dtype=np.float64)
FAIL_W /= FAIL_W.sum()
FAIL_LVS1 = np.arange(30, 40)  # 1성 실패 복귀 30~39강
FAIL_LVS2 = np.arange(50, 60)  # 2성 실패 복귀 50~59강

print('테이블 빌드 완료')
print(f'  SUCC1 shape: {SUCC1.shape}  (4 프리셋 × 50 레벨)')
print(f'  SUCC2 shape: {SUCC2.shape}  (4 프리셋 × 80 레벨)')
print(f'  SUCC3 shape: {SUCC3.shape}  (4 프리셋 × 100 레벨)')

In [ ]:
# ============================================================
# 3. GPU 배치 마르코프 체인 + 피트니스 함수
# ============================================================

def _batch_T(succ_tbl, fail_tbl, cost_base, pidx, cmult, xp):
    """
    B개 파라미터 조합을 동시에 마르코프 체인 계산.
    pidx:  (B,) int  — 확률 프리셋 인덱스
    cmult: (B,) float — 비용 배율
    반환:  (B, L) float — T[n] 값
    """
    B = len(pidx)
    L = cost_base.shape[0]
    # 비용 행렬: (B, L)
    C = cost_base[xp.newaxis, :] * cmult[:, xp.newaxis]
    # 확률 행렬: (B, L)
    S = succ_tbl[pidx, :]
    F = fail_tbl[pidx, :]
    T = xp.empty((B, L), dtype=xp.float64)
    prev = xp.zeros(B, dtype=xp.float64)
    for n in range(L):
        T[:, n] = (C[:, n] + F[:, n] * prev) / S[:, n]
        prev = T[:, n].copy()
    return T

def _batch_promo(T, fail_lvs, fail_w, pc, pr, xp):
    """
    성급강화 기대비용 배치 계산.
    T:       (B, L)
    fail_lvs: 실패 복귀 레벨 배열
    fail_w:   가중치
    pc, pr:  (B,) 비용/성공률
    """
    # cumrev[:,k] = T[:,k] + T[:,k+1] + ... + T[:,-1]
    cumrev = xp.cumsum(T[:, ::-1], axis=1)[:, ::-1]  # (B, L)
    # 실패 복귀 비용의 기대값: sum_k(w_k * cumrev[:,fail_lvs[k]])
    fail_cost = xp.sum(
        fail_w[xp.newaxis, :] * cumrev[:, fail_lvs],
        axis=1
    )  # (B,)
    return (pc + (1.0 - pr) * fail_cost) / pr

def batch_eval(idx, scores, xp):
    """
    B개 파라미터 조합 동시 평가.
    idx:    (B, 11) int — 파라미터 인덱스
    scores: (N,) float  — 내림차순 랭킹 점수
    xp:     numpy or cupy

    반환 (모두 numpy):
      fitness(B,), ex1(B,), ex2e(B,), ex2_80(B,), ex3e(B,), ex3(B,),
      n1(B,), n2(B,), n3(B,)
    """
    # ── 파라미터 추출 ─────────────────────────────────────────────
    c1_np   = COST1_OPTS[idx[:, 0]]
    c2_np   = COST2_OPTS[idx[:, 1]]
    c3_np   = COST3_OPTS[idx[:, 2]]
    pc12_np = PROMO12C[idx[:, 3]]
    pr12_np = PROMO12R[idx[:, 4]]
    pc23_np = PROMO23C[idx[:, 5]]
    pr23_np = PROMO23R[idx[:, 6]]
    p1_np   = idx[:, 7]
    p2_np   = idx[:, 8]
    p3_np   = idx[:, 9]

    # ── GPU/CPU 배열 변환 ─────────────────────────────────────────
    if USE_GPU and xp is not np:
        c1   = xp.array(c1_np);   c2  = xp.array(c2_np);  c3  = xp.array(c3_np)
        pc12 = xp.array(pc12_np); pr12 = xp.array(pr12_np)
        pc23 = xp.array(pc23_np); pr23 = xp.array(pr23_np)
        p1   = xp.array(p1_np);   p2  = xp.array(p2_np);  p3  = xp.array(p3_np)
        S1   = xp.array(SUCC1);   F1  = xp.array(FAIL1);  B1  = xp.array(BASE1)
        S2   = xp.array(SUCC2);   F2  = xp.array(FAIL2);  B2  = xp.array(BASE2)
        S3   = xp.array(SUCC3);   F3  = xp.array(FAIL3);  B3  = xp.array(BASE3)
        FW   = xp.array(FAIL_W)
        FL1  = xp.array(FAIL_LVS1)
        FL2  = xp.array(FAIL_LVS2)
    else:
        c1=c1_np; c2=c2_np; c3=c3_np
        pc12=pc12_np; pr12=pr12_np; pc23=pc23_np; pr23=pr23_np
        p1=p1_np; p2=p2_np; p3=p3_np
        S1=SUCC1; F1=FAIL1; B1=BASE1
        S2=SUCC2; F2=FAIL2; B2=BASE2
        S3=SUCC3; F3=FAIL3; B3=BASE3
        FW=FAIL_W; FL1=FAIL_LVS1; FL2=FAIL_LVS2

    # ── 마르코프 체인 배치 계산 ───────────────────────────────────
    T1 = _batch_T(S1, F1, B1, p1, c1, xp)
    T2 = _batch_T(S2, F2, B2, p2, c2, xp)
    T3 = _batch_T(S3, F3, B3, p3, c3, xp)

    ex1       = T1.sum(axis=1)
    pe12      = _batch_promo(T1, FL1, FW, pc12, pr12, xp)
    ex2_entry = ex1 + pe12
    ex2_80    = ex2_entry + T2.sum(axis=1)
    pe23      = _batch_promo(T2, FL2, FW, pc23, pr23, xp)
    ex3_entry = ex2_80 + pe23
    ex3       = ex3_entry + T3.sum(axis=1)

    # ── CPU로 전환 ────────────────────────────────────────────────
    def to_np(arr):
        if USE_GPU and xp is not np:
            return xp.asnumpy(arr)
        return np.asarray(arr)

    ex1_n  = to_np(ex1)
    ex2e_n = to_np(ex2_entry)
    ex2_n  = to_np(ex2_80)
    ex3e_n = to_np(ex3_entry)
    ex3_n  = to_np(ex3)

    # ── 도달 인원 (브로드캐스팅) ──────────────────────────────────
    # (N, 1) >= (1, B) → (N, B) → .sum(0) = (B,)
    sc = np.asarray(scores)
    n1 = (sc[:, np.newaxis] >= ex1_n[np.newaxis, :]).sum(axis=0)
    n2 = (sc[:, np.newaxis] >= ex2_n[np.newaxis, :]).sum(axis=0)
    n3 = (sc[:, np.newaxis] >= ex3_n[np.newaxis, :]).sum(axis=0)

    # ── 피트니스 계산 ─────────────────────────────────────────────
    # 목표: n1≈50, n2≈10, n3≈2 (3성 달성 가능)
    eps = 1e-9

    def lsq(a, t):
        """로그 스케일 오차²"""
        return np.log(np.maximum(a.astype(np.float64), eps) / t) ** 2

    err_n    = 1.0*lsq(n1,50) + 2.0*lsq(n2,10) + 4.0*lsq(n3,2)
    r12      = ex2_n / np.maximum(ex1_n, eps)
    r23      = ex3_n / np.maximum(ex2_n, eps)
    err_r    = (np.log(np.maximum(r12,eps)/4.0)**2 +
                np.log(np.maximum(r23,eps)/5.0)**2)
    err_hard = (np.maximum(ex3_n - 200000.0, 0) / 50000.0) ** 2

    fitness = err_n + 0.5*err_r + 2.0*err_hard

    # 하드 제약: n3 < 1이면 완전 제거
    fitness[n3 < 1] = 1e9

    return fitness, ex1_n, ex2e_n, ex2_n, ex3e_n, ex3_n, n1, n2, n3

# ── 빠른 검증 ────────────────────────────────────────────────────────
test_idx = np.array([[1,0,1, 1,1, 2,2, 1,1,1, 1]])  # 임의 파라미터 1개
fit_t, ex1_t, ex2e_t, ex2_t, ex3e_t, ex3_t, n1_t, n2_t, n3_t = batch_eval(test_idx, SCORES, xp)
print('함수 검증 OK')
print(f'  테스트 점수: {fit_t[0]:.4f}')
print(f'  E[1성 50강]:  {ex1_t[0]:>10,.0f}P')
print(f'  E[2성 80강]:  {ex2_t[0]:>10,.0f}P')
print(f'  E[3성 100강]: {ex3_t[0]:>10,.0f}P')
print(f'  도달 인원: 1성={n1_t[0]}, 2성={n2_t[0]}, 3성={n3_t[0]}')

In [ ]:
# ============================================================
# 4. CEM (Cross-Entropy Method) 변분추론 탐색
# ============================================================
#
# 작동 원리:
#   각 파라미터를 범주형 확률분포로 표현하고,
#   엘리트 샘플로 분포를 반복 업데이트하여 수렴.
#   → 랜덤 탐색 대비 훨씬 적은 평가 횟수로 최적값 도달

# ── 설정 ─────────────────────────────────────────────────────────────
# GPU 사용 여부에 따라 배치 크기 자동 조정
BATCH_SIZE  = 100_000 if USE_GPU else 5_000   # 1회 평가 조합 수
N_ITERS     = 300     if USE_GPU else 30       # CEM 반복 횟수
ELITE_FRAC  = 0.10                             # 엘리트 비율 (상위 10%)
ALPHA       = 0.70                             # 분포 업데이트 학습률
SMOOTH      = 0.01                             # 라플라스 스무딩
TOP_K       = 200                              # 최종 보관 후보 수
PRINT_EVERY = 10                               # 로그 출력 주기 (iter)

total_evals = BATCH_SIZE * N_ITERS
print(f'CEM 설정')
print(f'  배치 크기:   {BATCH_SIZE:>10,}')
print(f'  반복 횟수:   {N_ITERS:>10,}')
print(f'  총 평가 수:  {total_evals:>10,}  (≈{total_evals/1e6:.1f}억회 상당)')
print(f'  엘리트 비율: {ELITE_FRAC*100:.0f}%')
print(f'  학습률:      {ALPHA}')

# ── CEM 루프 ─────────────────────────────────────────────────────────
# 초기: 모든 파라미터 균등 분포
probs = [np.ones(s, dtype=np.float64) / s for s in SIZES]

best_sc   = float('inf')
best_row  = None
all_top   = []   # 상위 후보 수집
history   = []   # 반복별 최고 점수 기록

t0 = time.time()
print(f'\n탐색 시작...')

for it in range(N_ITERS):
    # ── 샘플링: 현재 분포에서 BATCH_SIZE개 조합 생성 ──────────────
    idx = np.column_stack([
        np.random.choice(SIZES[j], size=BATCH_SIZE, p=probs[j])
        for j in range(N_PARAMS)
    ])  # (BATCH_SIZE, N_PARAMS)

    # ── GPU 배치 평가 ────────────────────────────────────────────
    fit, ex1, ex2e, ex2_80, ex3e, ex3, n1, n2, n3 = batch_eval(idx, SCORES, xp)

    # ── 유효 후보 선별 (n3 >= 1 보장됨, fit < 1e8) ──────────────
    vi   = np.where(fit < 1e8)[0]   # 유효 위치
    nv   = len(vi)

    if nv > 0:
        # 유효 후보를 점수 기준 정렬
        order_v     = np.argsort(fit[vi])          # vi 내 정렬
        ne          = max(1, int(nv * ELITE_FRAC))  # 엘리트 수
        elite_pos   = vi[order_v[:ne]]             # 엘리트의 원본 위치

        # ── 분포 업데이트 ─────────────────────────────────────────
        for j in range(N_PARAMS):
            cnt = np.bincount(idx[elite_pos, j],
                              minlength=SIZES[j]).astype(np.float64)
            cnt += SMOOTH
            new_p = cnt / cnt.sum()
            blended = ALPHA * new_p + (1.0 - ALPHA) * probs[j]
            probs[j] = blended / blended.sum()

        # ── 상위 후보 수집 ────────────────────────────────────────
        for k in range(ne):
            pos = elite_pos[k]
            sc  = float(fit[pos])
            all_top.append({
                'score':   sc,
                'row':     idx[pos].copy(),
                'ex1':     float(ex1[pos]),
                'ex2e':    float(ex2e[pos]),
                'ex2_80':  float(ex2_80[pos]),
                'ex3e':    float(ex3e[pos]),
                'ex3':     float(ex3[pos]),
                'n1':      int(n1[pos]),
                'n2':      int(n2[pos]),
                'n3':      int(n3[pos]),
            })
            if sc < best_sc:
                best_sc  = sc
                best_row = idx[pos].copy()

    history.append(best_sc)

    if (it + 1) % PRINT_EVERY == 0 or it == 0:
        elapsed = time.time() - t0
        print(f'  [{it+1:3d}/{N_ITERS}] '
              f'유효 {nv:6,}/{BATCH_SIZE:,} '
              f'| 최고 {best_sc:.4f} '
              f'| {elapsed:6.1f}s '
              f'| {BATCH_SIZE*(it+1)/elapsed/1e6:.2f}M eval/s')

total_t = time.time() - t0
print(f'\n탐색 완료!')
print(f'  총 소요: {total_t:.1f}s')
print(f'  총 평가: {BATCH_SIZE*N_ITERS:,}회')
print(f'  처리 속도: {BATCH_SIZE*N_ITERS/total_t/1e6:.2f}M eval/s')

In [ ]:
# ============================================================
# 5. 결과 정리 및 출력
# ============================================================

# 중복 제거 & 정렬
seen, dedup = set(), []
for c in sorted(all_top, key=lambda x: x['score']):
    key = tuple(c['row'])
    if key not in seen:
        seen.add(key)
        dedup.append(c)
candidates = dedup[:TOP_K]

print(f'상위 {len(candidates)}개 후보 (중복 제거, 점수 낮을수록 좋음)\n')

if not candidates:
    print('유효 후보 없음 — 파라미터 범위 또는 피트니스 설정을 확인하세요')
else:
    best = candidates[0]
    p    = best['row']

    print('=' * 55)
    print('  BEST CANDIDATE (v3: CEM 변분추론, 3성 달성 가능)')
    print('=' * 55)
    print(f'  score: {best["score"]:.4f}\n')

    print('[파라미터]')
    print(f'  일반강화 비용 배율: 1성 ×{COST1_OPTS[p[0]]}, '
          f'2성 ×{COST2_OPTS[p[1]]}, '
          f'3성 ×{COST3_OPTS[p[2]]}')
    print(f'  확률 프리셋: 1성={PRESET_NAMES[p[7]]}, '
          f'2성={PRESET_NAMES[p[8]]}, '
          f'3성={PRESET_NAMES[p[9]]}')
    print(f'  성급강화 1→2: {PROMO12C[p[3]]:.0f}P / {PROMO12R[p[4]]*100:.0f}%')
    print(f'  성급강화 2→3: {PROMO23C[p[5]]:.0f}P / {PROMO23R[p[6]]*100:.0f}%')
    print(f'  보너스 프로필: {BONUS_NAMES[p[10]]}')

    print('\n[기대비용 (누적)]')
    print(f'  E[1성 50강]:  {best["ex1"]:>12,.0f}P')
    print(f'  E[2성 진입]:  {best["ex2e"]:>12,.0f}P')
    print(f'  E[2성 80강]:  {best["ex2_80"]:>12,.0f}P')
    print(f'  E[3성 진입]:  {best["ex3e"]:>12,.0f}P')
    print(f'  E[3성 100강]: {best["ex3"]:>12,.0f}P')

    print('\n[도달 가능 인원 (실제 Production 데이터)]')
    print(f'  1성 50강: {best["n1"]}명')
    print(f'  2성 80강: {best["n2"]}명')
    print(f'  3성 100강: {best["n3"]}명')

    print('\n--- 상위 10개 요약 ---')
    header = f'{"순위":>4} {"score":>8} {"비용1/2/3":>15} {"확률1/2/3":>20} '\
             f'{"성급12":>11} {"성급23":>12} {"E[3성]",">12"} {"n1/n2/n3":>10}'
    print(f'{"".join(header)}')
    for i, c in enumerate(candidates[:10]):
        q = c['row']
        cst = f'{COST1_OPTS[q[0]]}/{COST2_OPTS[q[1]]}/{COST3_OPTS[q[2]]}'
        prs = f'{PRESET_NAMES[q[7]][:3]}/{PRESET_NAMES[q[8]][:3]}/{PRESET_NAMES[q[9]][:3]}'
        p12 = f'{PROMO12C[q[3]]:.0f}P/{PROMO12R[q[4]]*100:.0f}%'
        p23 = f'{PROMO23C[q[5]]:.0f}P/{PROMO23R[q[6]]*100:.0f}%'
        ns  = f'{c["n1"]}/{c["n2"]}/{c["n3"]}'
        print(f'{i+1:>4} {c["score"]:>8.4f} {cst:>15} {prs:>20} '
              f'{p12:>11} {p23:>12} {c["ex3"]:>12,.0f} {ns:>10}')

In [ ]:
# ============================================================
# 6. 결과 CSV 저장 및 다운로드
# ============================================================

rows = []
for c in candidates:
    q = c['row']
    rows.append({
        'score':         c['score'],
        'c1_mult':       COST1_OPTS[q[0]],
        'c2_mult':       COST2_OPTS[q[1]],
        'c3_mult':       COST3_OPTS[q[2]],
        'promo12_cost':  PROMO12C[q[3]],
        'promo12_rate':  PROMO12R[q[4]],
        'promo23_cost':  PROMO23C[q[5]],
        'promo23_rate':  PROMO23R[q[6]],
        'prob1':         PRESET_NAMES[q[7]],
        'prob2':         PRESET_NAMES[q[8]],
        'prob3':         PRESET_NAMES[q[9]],
        'bonus':         BONUS_NAMES[q[10]],
        'ex_1_50':       c['ex1'],
        'ex_2_entry':    c['ex2e'],
        'ex_2_80':       c['ex2_80'],
        'ex_3_entry':    c['ex3e'],
        'ex_3_100':      c['ex3'],
        'n_1_50':        c['n1'],
        'n_2_80':        c['n2'],
        'n_3_100':       c['n3'],
    })

df = pd.DataFrame(rows)
df.index += 1
csv_path = 'artifact_params_v3_CEM.csv'
df.to_csv(csv_path, index_label='rank', encoding='utf-8-sig')
print(f'CSV 저장: {csv_path}')
print(df[['score','c1_mult','c2_mult','c3_mult','prob1','prob2','prob3',
          'promo12_cost','promo12_rate','promo23_cost','promo23_rate',
          'ex_1_50','ex_2_80','ex_3_100','n_1_50','n_2_80','n_3_100']]
      .head(20).to_string())

try:
    from google.colab import files
    files.download(csv_path)
    print('\nCSV 다운로드 시작')
except Exception:
    print(f'\n로컬 저장: {csv_path}')

In [ ]:
# ============================================================
# 7. 시각화
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('유물 파라미터 탐색 v3 (CEM 변분추론 + GPU)', fontsize=14, fontweight='bold')

top20 = candidates[:20]
rnks  = list(range(1, len(top20)+1))

# (1) 단계별 누적 기대비용
ax = axes[0, 0]
b1 = [c['ex1']/1000 for c in top20]
b2 = [(c['ex2_80'] - c['ex1'])/1000 for c in top20]
b3 = [(c['ex3'] - c['ex2_80'])/1000 for c in top20]
ax.bar(rnks, b1, label='1성 완성', color='#5B9BD5', alpha=0.85)
ax.bar(rnks, b2, bottom=b1, label='2성 추가', color='#ED7D31', alpha=0.85)
ax.bar(rnks, b3, bottom=[a+b for a,b in zip(b1,b2)], label='3성 추가', color='#A9D18E', alpha=0.85)
ax.set_xlabel('순위')
ax.set_ylabel('누적 E[X] (천P)')
ax.set_title('단계별 누적 기대비용')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# (2) 도달 가능 인원
ax = axes[0, 1]
ax.plot(rnks, [c['n1'] for c in top20], 'o-', label='n_1_50 (1성 완성)', color='#5B9BD5')
ax.plot(rnks, [c['n2'] for c in top20], 's-', label='n_2_80 (2성 완성)', color='#ED7D31')
ax.plot(rnks, [c['n3'] for c in top20], '^-', label='n_3_100 (3성 완성)', color='#A9D18E', linewidth=2)
ax.axhline(y=1, color='red', linestyle='--', alpha=0.6, label='3성 최소 1명')
ax.axhline(y=2, color='orange', linestyle=':', alpha=0.6, label='3성 목표 2명')
ax.set_xlabel('순위')
ax.set_ylabel('도달 가능 인원')
ax.set_title('단계별 도달 인원')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (3) CEM 수렴 곡선
ax = axes[1, 0]
valid_hist = [h for h in history if h < 1e8]
if valid_hist:
    ax.plot(range(1, len(history)+1), [min(h, 20) for h in history], 'b-', linewidth=1.5)
    ax.axhline(y=valid_hist[-1], color='red', linestyle='--', alpha=0.7,
               label=f'최종 {valid_hist[-1]:.4f}')
ax.set_xlabel('반복 (iteration)')
ax.set_ylabel('최고 피트니스 점수')
ax.set_title('CEM 수렴 곡선 (낮을수록 좋음)')
ax.legend()
ax.grid(alpha=0.3)

# (4) E[3성] vs n_3_100 산포도
ax = axes[1, 1]
ex3_vals = [c['ex3'] for c in candidates]
n3_vals  = [c['n3']  for c in candidates]
scores_c = [c['score'] for c in candidates]
sc_norm  = plt.Normalize(min(scores_c), max(scores_c))
sc_map   = ax.scatter(ex3_vals, n3_vals, c=scores_c, cmap='RdYlGn_r',
                      norm=sc_norm, s=40, alpha=0.7)
plt.colorbar(sc_map, ax=ax, label='피트니스 점수')
if candidates:
    ax.scatter([candidates[0]['ex3']], [candidates[0]['n3']],
               color='red', s=120, zorder=5, label=f'Best\n({candidates[0]["ex3"]:,.0f}P, {candidates[0]["n3"]}명)')
ax.set_xlabel('E[3성 완성] (P)')
ax.set_ylabel('3성 도달 인원')
ax.set_title('E[3성] vs 도달 인원')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
chart_path = 'artifact_params_v3_chart.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'차트 저장: {chart_path}')

try:
    from google.colab import files
    files.download(chart_path)
except Exception:
    pass

In [ ]:
# ============================================================
# 8. 수렴 분포 확인 (최종 파라미터 분포 시각화)
# ============================================================
# CEM이 어떤 파라미터 값으로 수렴했는지 분포 확인

param_labels = [
    ('1성 비용 배율', COST1_OPTS),
    ('2성 비용 배율', COST2_OPTS),
    ('3성 비용 배율', COST3_OPTS),
    ('성급1→2 비용', PROMO12C),
    ('성급1→2 성공률', PROMO12R),
    ('성급2→3 비용', PROMO23C),
    ('성급2→3 성공률', PROMO23R),
    ('확률 프리셋(1성)', np.array(PRESET_NAMES)),
    ('확률 프리셋(2성)', np.array(PRESET_NAMES)),
    ('확률 프리셋(3성)', np.array(PRESET_NAMES)),
    ('보너스 프로필', np.array(BONUS_NAMES)),
]

fig, axes2 = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('CEM 수렴 후 파라미터 분포', fontsize=13, fontweight='bold')
axes2_flat = axes2.flatten()

for j, (label, opts) in enumerate(param_labels):
    ax = axes2_flat[j]
    p  = probs[j]
    opt_strs = [str(o) for o in opts]
    bars = ax.bar(range(len(p)), p * 100, color='#5B9BD5', alpha=0.8)
    ax.set_title(label, fontsize=9)
    ax.set_xticks(range(len(p)))
    ax.set_xticklabels(opt_strs, fontsize=7, rotation=30, ha='right')
    ax.set_ylabel('확률 (%)', fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    # 최대 확률 강조
    best_idx = np.argmax(p)
    bars[best_idx].set_color('#C00000')

# 빈 칸 숨기기
for j in range(len(param_labels), len(axes2_flat)):
    axes2_flat[j].set_visible(False)

plt.tight_layout()
dist_path = 'artifact_params_v3_dist.png'
plt.savefig(dist_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'분포 차트 저장: {dist_path}')

try:
    from google.colab import files
    files.download(dist_path)
except Exception:
    pass

print('\n모든 작업 완료!')